# Data-only semantic guidance research checks
Grain: one loaded visual-obligation profile.
Counts describe authored metadata, not knowledge gaps, censorship, or image success.
Selected profiles are a purposive review sample, not an estimate of dataset-wide quality.


In [1]:
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone, timedelta
import hashlib, json, sys
root = Path.cwd()
skill = root / "skills/photo-prompt-image-generator"
sys.path.insert(0, str(skill / "scripts"))
import prompt_generator as pg
profiles = pg.load_visual_obligation_registry(skill / "assets/photo_prompt_visual_obligations.json")["profiles"]
by_id = {p["id"]: p for p in profiles}
counts = {
    "profiles": len(profiles),
    "unique_profile_ids": len(by_id),
    "runtime_expression_default_modes": dict(Counter(p["runtime_expression"].get("default_mode") for p in profiles)),
    "profiles_with_context_disambiguation": sum(bool(p["activation"].get("context_disambiguation")) for p in profiles),
    "profiles_with_runtime_forbidden_labels": sum(bool(p["runtime_expression"].get("runtime_forbidden_labels")) for p in profiles),
}
restricted = [p for p in profiles if p["runtime_expression"].get("runtime_forbidden_labels")]
counts["restricted_profiles_by_category"] = dict(Counter(p["category"] for p in restricted))
counts["restricted_profiles_by_default_mode"] = dict(Counter(p["runtime_expression"].get("default_mode") for p in restricted))
pfe_raw = json.loads((skill / "assets/photo_prompt_visual_obligations_portrait_fashion_exposure.json").read_text())
pfe_ids = {p["id"] for p in pfe_raw["profiles"]}
pfe = [p for p in profiles if p["id"] in pfe_ids]
counts["portrait_fashion_extension"] = {
    "profiles": len(pfe),
    "default_modes": dict(Counter(p["runtime_expression"].get("default_mode") for p in pfe)),
    "nonempty_runtime_forbidden_labels": sum(bool(p["runtime_expression"].get("runtime_forbidden_labels")) for p in pfe),
    "nonempty_forbidden_prompt_terms": sum(bool(p["runtime_expression"].get("forbidden_prompt_terms")) for p in pfe),
    "nonempty_prompt_label_terms": sum(bool(p["runtime_expression"].get("prompt_label_terms")) for p in pfe),
}
print(json.dumps(counts, ensure_ascii=False, indent=2))

{
  "profiles": 1385,
  "unique_profile_ids": 1385,
  "runtime_expression_default_modes": {
    "label_plus_definition": 200,
    "definition_only": 40,
    "definition_with_optional_label": 1145
  },
  "profiles_with_context_disambiguation": 54,
  "profiles_with_runtime_forbidden_labels": 41,
  "restricted_profiles_by_category": {
    "composite_expression": 1,
    "affective_self_presentation": 1,
    "behavioral_archetype": 3,
    "adult_fantasy_garment_mixin": 1,
    "adult_body_build_relation": 1,
    "adult_regional_body_relation": 1,
    "adult_appeal_relational_display": 3,
    "adult_motion_display": 1,
    "adult_environmental_appeal_display": 1,
    "facial_shape_relation": 3,
    "makeup_layer_balance": 2,
    "complexion_optics": 1,
    "cheek_color_distribution": 1,
    "hair_lock_topology": 1,
    "luxury_visual_grammar": 1,
    "luxury_object_construction": 1,
    "luxury_service_interaction": 1,
    "historical_private_household_spatial_relation": 1,
    "historical_sc

In [2]:
selected_ids = [
    "aircraft_pilot_operation", "one_piece_dress_construction",
    "embodied_corruption_transition", "kuudere_composed_warmth_relation",
    "pfe_cowl", "clothing_ct037_v1", "pfe_one_shoulder", "pfe_ruching",
    "sheer_garment_optical_layering", "clothing_ct023_v2", "clothing_ct090_v2",
    "split_diopter_dual_focus_planes", "film_halation_highlight_edge_relation",
    "contrapposto_weight_shift", "tribhanga_three_bend_pose",
    "broad_face_light_orientation_relation", "short_face_light_orientation_relation",
    "cr_warm_foreground", "cr_cool_foreground",
]
selected = []
for ident in selected_ids:
    p = by_id[ident]
    semantics = p["semantics"]
    selected.append({
        "id": ident,
        "definition": semantics["definition"],
        "paraphrase_examples": semantics.get("paraphrase_examples", []),
        "contrast_examples": semantics.get("contrast_examples", []),
        "context_disambiguation": p["activation"].get("context_disambiguation"),
        "default_mode": p["runtime_expression"].get("default_mode"),
        "authored_components": p.get("authored_components", []),
        "claim_limits": semantics.get("claim_limits", []),
    })
print(json.dumps({"selected_existing_profiles": len(selected), "ids": selected_ids}, ensure_ascii=False, indent=2))

{
  "selected_existing_profiles": 19,
  "ids": [
    "aircraft_pilot_operation",
    "one_piece_dress_construction",
    "embodied_corruption_transition",
    "kuudere_composed_warmth_relation",
    "pfe_cowl",
    "clothing_ct037_v1",
    "pfe_one_shoulder",
    "pfe_ruching",
    "sheer_garment_optical_layering",
    "clothing_ct023_v2",
    "clothing_ct090_v2",
    "split_diopter_dual_focus_planes",
    "film_halation_highlight_edge_relation",
    "contrapposto_weight_shift",
    "tribhanga_three_bend_pose",
    "broad_face_light_orientation_relation",
    "short_face_light_orientation_relation",
    "cr_warm_foreground",
    "cr_cool_foreground"
  ]
}


In [3]:
manifest_path = root / "docs/analysis/2026-10-02-visual-semantics-data-audit/source-manifest.json"
manifest = json.loads(manifest_path.read_text())
source_files = []
for entry in manifest["files"]:
    data = (root / entry["path"]).read_bytes()
    observed = hashlib.sha256(data).hexdigest()
    source_files.append({"path": entry["path"], "sha256": observed, "bytes": len(data), "matches_prior_audit": observed == entry["sha256"]})
integrity = {
    "source_files_checked": len(source_files),
    "all_match_prior_audit": all(x["matches_prior_audit"] for x in source_files),
    "mismatches": [x["path"] for x in source_files if not x["matches_prior_audit"]],
    "prior_audit_commit": manifest["commit"],
}
print(json.dumps(integrity, ensure_ascii=False, indent=2))
assert integrity["all_match_prior_audit"], "Source drift requires re-evaluation"

{
  "source_files_checked": 30,
  "all_match_prior_audit": true,
  "mismatches": [],
  "prior_audit_commit": "0e5cc0d1968d5cfe83d1b63e9cfa198724ea9463"
}
